# Stage 06 — Data Preprocessing

Stage 06. **I based it on the lecture notebook.**

In [1]:
# Install missing packages (uncomment and run to install).
# !pip install numpy pandas

In [2]:
from pathlib import Path

# Project root.
ROOT = Path.cwd()
if not (ROOT/"src/cleaning.py").exists() and (ROOT.parent/"src/cleaning.py").exists():
    ROOT = ROOT.parent

CHECKS = [
    ("src/cleaning.py", "NEEDED", "fill, drop, and normalize helpers"),
    ("data/raw/sample_data.csv", "NEEDED", "raw sample with missing values"),
]

print(f"Looking in: {ROOT}\n")
missing = 0
for rel, kind, note in CHECKS:
    here = (ROOT/rel).exists()
    if not here and kind == "NEEDED":
        missing += 1
    print(f"  [{'OK ' if here else 'MISS'}]  {kind:<8}  {rel:<34}  {note}")

if missing:
    raise FileNotFoundError(f"{missing} needed file(s) missing under {ROOT}")
print("\nAll needed files present.")

Looking in: /Users/ghostof0days/projects/bootcamp/homework/homework06

  [OK ]  NEEDED    src/cleaning.py                     fill, drop, and normalize helpers
  [OK ]  NEEDED    data/raw/sample_data.csv            raw sample with missing values

All needed files present.


In [3]:
import sys

import pandas as pd

# Import helpers from src/.
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.cleaning import drop_missing, fill_missing_median, normalize_data

RAW = ROOT/"data/raw"
PROC = ROOT/"data/processed"
PROC.mkdir(parents=True, exist_ok=True)

## Load raw dataset

In [4]:
sample = pd.read_csv(RAW/"sample_data.csv")
print("Original shape:", sample.shape)
print("Original NA count:\n", sample.isna().sum())
sample.head()

Original shape: (7, 6)
Original NA count:
 age           1
income        3
score         1
zipcode       0
city          0
extra_data    5
dtype: int64


,age,income,score,zipcode,city,extra_data
0,34.0,55000.0,0.82,90210,Beverly,NaN
1,45.0,NaN,0.91,10001,New York,42.0
2,29.0,42000.0,NaN,60614,Chicago,NaN
3,50.0,58000.0,0.76,94103,SF,NaN
4,38.0,NaN,0.88,73301,Austin,NaN


## Apply cleaning functions

In [5]:
numeric_cols = ["age", "income", "score"]

# Fill numeric gaps.
filled = fill_missing_median(sample, numeric_cols)

# Drop mostly-empty columns.
dropped = drop_missing(filled, threshold=0.5)

# Scale remaining numeric columns.
cleaned = normalize_data(dropped, numeric_cols)

print("Cleaned shape:", cleaned.shape)
print("Cleaned NA count:\n", cleaned.isna().sum())
print("\nOriginal numeric statistics:")
print(sample[numeric_cols].describe())
print("\nCleaned numeric statistics:")
print(cleaned[numeric_cols].describe())
cleaned.head()

Cleaned shape: (7, 5)
Cleaned NA count:
 age        0
income     0
score      0
zipcode    0
city       0
dtype: int64

Original numeric statistics:
             age        income     score
count   6.000000      4.000000  6.000000
mean   39.500000  51000.000000  0.801667
std     7.556454   7071.067812  0.092826
min    29.000000  42000.000000  0.650000
25%    35.000000  47250.000000  0.767500
50%    39.500000  52000.000000  0.805000
75%    44.000000  55750.000000  0.865000
max    50.000000  58000.000000  0.910000

Cleaned numeric statistics:
            age    income     score
count  7.000000  7.000000  7.000000
mean   0.500000  0.589286  0.585165
std    0.328479  0.314281  0.325952
min    0.000000  0.000000  0.000000
25%    0.333333  0.531250  0.480769
50%    0.500000  0.625000  0.596154
75%    0.666667  0.718750  0.769231
max    1.000000  1.000000  1.000000


,age,income,score,zipcode,city
0,0.238095,0.8125,0.653846,90210,Beverly
1,0.761905,0.6250,1.000000,10001,New York
2,0.000000,0.0000,0.596154,60614,Chicago
3,1.000000,1.0000,0.423077,94103,SF
4,0.428571,0.6250,0.884615,73301,Austin


## Save cleaned dataset

In [6]:
cleaned_path = PROC/"sample_data_cleaned.csv"
cleaned.to_csv(cleaned_path, index=False)
print("Saved cleaned CSV:", cleaned_path)

Saved cleaned CSV: /Users/ghostof0days/projects/bootcamp/homework/homework06/data/processed/sample_data_cleaned.csv


## Documentation

- I filled `age`, `income`, and `score` with each column's median.
- I dropped columns whose NA share is above 0.5, which removes `extra_data`.
- I min-max scaleD those three numeric columns to [0, 1].
- I saveD the result to `data/processed/sample_data_cleaned.csv`.
- I added the README Cleaning Strategy section in `homework/homework06/README.md`.

Assumptions and risks:
- When I median fill, I assumed the missing numbers are not systematically biased due to the MCAR or MAR concepts from lecture.
- I don't need the columns in `extra_data` later.
- When I min-max scaled, I assumed that the observed min and max are representative.